In [ ]:
!python -m pip install scikit-learn
!env\Scripts\Activate.ps1 



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# imports 
import pandas as pd 
import scikit-learn


SyntaxError: invalid syntax (3022832359.py, line 3)

In [7]:
tf.config.list_physical_devices('GPU')

NameError: name 'tf' is not defined

# Dataset loading and preprocessing


# Kalman (SELF-CARE) ensemble
## 1. Data Collection & Setup
Devices: wrist‑ or finger‑mounted GSR, PPG, and skin‑temperature sensors.

Protocol: record during your baseline vs. stress (and any other) tasks, ensuring you log a synchronized timestamp and a ground‑truth label stream.

Storage: keep each subject’s raw streams (timestamp + value) in a uniform format (e.g. one CSV or pickle per subject).

## 2. Preprocessing
Resample & Align

Choose a reference rate (e.g. your highest‑rate sensor) and up/down‑sample the others so all three share a common timestamp index.

Filtering

GSR: low‑pass Butterworth (∼1 Hz) to remove high‑frequency noise; optionally run cvxEDA to split phasic/tonic components.

PPG: band‑pass (0.5–8 Hz) to isolate the pulse waveform; remove baseline wander.

TEMP: low‑pass (∼0.5 Hz) to smooth out rapid fluctuations.

Windowing

Segment into fixed windows (e.g. 1 s non‑overlapping or 50 % overlap).

Assign each window a label (e.g. via majority vote or forward‑fill from your label stream).

## 3. Context Identification (Gating)
Since you don’t have ACC, use a signal‑quality or motion proxy derived from your sensors—for example:

PPG SQI: compute the ratio of high‑ to low‑frequency power, or the standard deviation of beat‑to‑beat intervals.

GSR Variability: the variance or spectral entropy of the tonic component.

Extract SQI Features per window.

Train a Decision Tree on these SQIs to predict which “branch” (sensor subset) to run.

Performance‑Energy Trade‑off (δ)

Let b̄ = branch with highest gating probability.

Activate any branch with prob ≥ b̄ − δ (δ∈[0,1] controls how many extra branches you’ll allow).

## 4. Branch Classifier Training
Define Your Branches (early‑fusion subsets), e.g.:

B1 = {GSR, PPG, TEMP}

B2 = {GSR, PPG}

B3 = {GSR, TEMP}

B4 = {PPG, TEMP}

Feature Extraction

For each window & each branch, compute summary stats (mean, std, min, max) on each modality + any domain‑specific features (e.g. PPG peak frequency, GSR phasic amplitude, TEMP slope).

Classifier Selection

For each branch, train a small suite of models (e.g. DT, RF, AB, LDA, KNN) via leave‑one‑subject‑out CV.

Pick the best‑performing classifier per branch (lowest validation loss).

5. Late Fusion with Kalman Filter
Initialize

State x₀ = your prior class‑prob vector (e.g. [0.8,0.1,0.1] for 3‑class), P₀ = small covariance.

Per‑Window Inference

Gating → select branch(es) → run their classifiers → get each’s zₜ (prob‑vector).

Threshold & Scale zₜ (ε filter + γ scaling to handle low confidence or class imbalance).

Kalman Predict/Update (A=H=I, tune process & measurement noise) to fuse zₜ into xₜ.

Output

Final stress prediction = arg max(xₜ).

6. Validation & Deployment
LOSO‑CV: test generalization across subjects.

Energy Profiling: measure per‑branch compute to pick δ.

Pipeline Packaging: wrap preprocessing → gating → branch inference → fusion into one reusable module or microservice.

Real‑Time: ensure each 1 s window (or your chosen window length) is fully processed before the next arrives.

With these tweaks—filtering GSR/PPG/TEMP, gating on signal‑quality proxies, and redefining your branches—you’ll have a SELF‑CARE–style stress detector built around your three sensors.

Source - https://ieeexplore.ieee.org/document/9881725

# CNN model with deep neural for classification 
🔧 Model Construction (CNN-Based Time-Series Classifier)
1. Why CNN for Stress Detection?
They’re using 1D CNNs to learn directly from raw sensor data (no manual feature extraction).

CNNs are well-suited for time-series tasks (like EDA, PPG, TEMP) because they:

Capture local temporal dependencies

Are scale invariant (pattern can shift in time)

2. Architecture Summary
🔷 Input
Raw time-series window from sensors (like EDA/GSR).

🔷 Convolutional Stack
Conv Layer 1:

100 filters, kernel size 5

Learns short-term temporal patterns (e.g., sudden changes in GSR)

Conv Layer 2:

100 filters, kernel size 10

Learns longer-range or more complex patterns

Global Max Pooling:

Reduces each feature map to a single value

Removes dependency on exact input length

Feature Output:

A flat vector of aggregated features across all kernels

🔷 Dense Stack
FC1: 128 neurons, ReLU, Dropout 0.3

FC2: 64 neurons, ReLU, Dropout 0.2

Output Layer: Softmax over stress classes (e.g., stress vs. no stress or multiple levels)

🧠 Activations
ReLU: All layers except output

Softmax: Final layer for multi-class stress classification

🧬 Why No Feature Engineering?
The CNN automatically learns hierarchical features from raw input:

Low-level edges, slopes → High-level temporal events

No need for handcrafted features like peaks, slopes, derivatives—just clean, structured raw data.

👤 Model Personalization (Online Learning Loop)
1. Problem:
Stress response is subjective → same signal ≠ same label across users.

A general model may underfit or misclassify personal stress triggers.

2. Solution: Online Fine-Tuning
M1 (General Model): Pretrained on population-level dataset

While in use:

Collect labeled user data (e.g., from self-report or proxy)

Fine-tune M1 on this personal data to get M2 (Personalized Model)

Iterate until M2 achieves satisfactory performance for the individual

3. Benefits
Adapts to each user’s baseline and unique stress patterns

Reduces generalization gap between training population and end-user

🚀 Deployment Insight (for you as an ML engineer)
If you were implementing this:

You’d stream EDA/PPG/TEMP data in real-time.

Chunk into fixed-length windows → feed to the CNN

The CNN model runs on-device (for latency/privacy) or server (if bandwidth permits)

You’d implement a lightweight fine-tuning pipeline, likely using only the dense layers (frozen CNN base) for quick personalization

Consider model checkpointing and gradual learning rate decay to avoid overfitting during personalization

Source - https://ieeexplore.ieee.org/document/9871842

# transformer

## 🧠 Architecture Flow
Input:

Multi-sensor time-series window (GSR, PPG, Temp)

Embed input: Linear layer or 1D Conv → project to d_model (e.g., 128)

Add positional encodings to preserve time order

Transformer Encoder Stack:

Multi-Head Self-Attention (e.g., 4–8 heads)

Feedforward layers (e.g., 256 units)

LayerNorm + Dropout

Repeat for N layers (e.g., 2–4)

Pooling:

Global Average or CLS token

Dense Stack:

FC1: 64 neurons

Dropout

Output: Softmax



# Temporal CN
🧠 Architecture Flow
Input: Raw GSR, PPG, Temp time-series

TCN Block(s):

Multiple layers of 1D Conv with increasing dilation rates (1, 2, 4, 8, …)

Causal padding to preserve time order

Residual connections for stable training

Each block = [Conv1D → ReLU → Dropout]

Flatten or Global Average Pooling

Dense Stack:

FC1: 64 neurons

Dropout

Output: Softmax



# 1D CNN + LSTM 
Input: Raw time-series window from sensors (GSR, PPG, Temp)

CNN Stack:

1D Conv Layer 1: 64 filters, kernel size 5 → learn short-term patterns

1D Conv Layer 2: 64 filters, kernel size 3 → refine features

Optional: BatchNorm, ReLU, Dropout

RNN Layer:

LSTM or GRU layer with 64-128 hidden units

Captures sequence-level memory (e.g., stress buildup)

Dense Stack:

FC1: 64 neurons, ReLU

Dropout

Output Layer: Softmax (for classification)